# Phase 3: 멀티에이전트 — 전문가 에이전트 분리 + 오케스트레이터

각 분석을 **독립적인 전문가 에이전트**로 분리하고,
**Supervisor(오케스트레이터)**가 동적으로 작업을 배분합니다.

### 이 Phase에서 배우는 것
- Supervisor 패턴 구현
- 에이전트 간 라우팅 (조건부 엣지)
- 멀티에이전트 그래프 구성

## 1. State 확장 — Supervisor 지원

## 2. Supervisor (오케스트레이터) 구현

## 3. 전문가 에이전트 노드 정의

## 4. Supervisor 멀티에이전트 그래프 구성

## 5. 재귀 가드 (recursion_limit)

### 왜 여기서 중요한가
Phase 3의 멀티에이전트 그래프는 **Supervisor가 조건부 라우팅**으로 다음 노드를 선택합니다.
Supervisor → Analyst → Supervisor → ... 순환 구조이므로, **라우팅 로직에 버그가 있거나
모델이 종결 판단을 내리지 못하면 무한 순환**이 발생합니다.

LangGraph는 기본 `recursion_limit = 25`가 암묵적으로 적용되지만, **25 hop 도달까지의
모든 LLM 호출은 과금**됩니다. 멀티에이전트는 hop당 호출 수가 높아(전문가별 분석 + 종합)
**25 hop = 50~100+ LLM 호출**이 되는 비용 폭증 경로입니다.

### 권장: 학습 환경에서 10 이하로 고정
정상 흐름에서 Phase 3 그래프는 5~7 hop이면 완료되므로, **10**이면 충분한 여유이며
루프 발생 시 조기 실패로 비용을 막을 수 있습니다.

```python
# 기존: 기본 recursion_limit=25에 암묵적 의존
result = multi_agent_graph.invoke({...})

# 권장: 명시적 상한
result = multi_agent_graph.invoke(
    {...},
    config={"recursion_limit": 10},
)
```

실패 시 `GraphRecursionError`가 발생합니다. 이는 버그 신호이므로 try/except로 삼키지 말고
Supervisor의 종결 조건(`route_supervisor`의 반환값)을 재검토하세요.


## Phase 3 완료

Supervisor가 동적으로 에이전트를 배분하는 멀티에이전트 시스템을 구축했습니다.

```
         ┌──────────┐
         │Supervisor│ ←────────────────────┐
         └────┬─────┘                       │
        ┌─────┼──────┬──────┬──────┐       │
        v     v      v      v      v       │
    [Research] [Quant] [Qual] [Risk] [Report]│
        │      │      │      │      │       │
        └──────┴──────┴──────┴──────┘───────┘
```

**다음 Phase**: Human-in-the-loop + 스트리밍 + 최종 리포트 개선